In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [2]:
df=pd.read_csv("skills2025.csv")
print(df.shape)

(65437, 114)


In [3]:
cols=["DevType", "Industry", "LanguageHaveWorkedWith", "LanguageWantToWorkWith", "LanguageAdmired", "DatabaseHaveWorkedWith", "DatabaseWantToWorkWith", "DatabaseAdmired", "PlatformHaveWorkedWith", "PlatformWantToWorkWith", "PlatformAdmired", "WebframeHaveWorkedWith", "WebframeWantToWorkWith", "WebframeAdmired", "EmbeddedHaveWorkedWith", "EmbeddedWantToWorkWith", "EmbeddedAdmired", "MiscTechHaveWorkedWith", "MiscTechWantToWorkWith", "MiscTechAdmired", "ToolsTechHaveWorkedWith", "ToolsTechWantToWorkWith", "ToolsTechAdmired", "NEWCollabToolsHaveWorkedWith", "NEWCollabToolsWantToWorkWith", "NEWCollabToolsAdmired", "OfficeStackAsyncHaveWorkedWith", "OfficeStackAsyncWantToWorkWith", "OfficeStackAsyncAdmired", "OfficeStackSyncHaveWorkedWith", "OfficeStackSyncWantToWorkWith", "OfficeStackSyncAdmired","YearsCode", "YearsCodePro", "WorkExp"]
df=df[cols]

In [4]:
df.dropna(subset=['DevType'],inplace=True)

In [5]:
null_list_temp=df.isnull().sum()
null_list_temp=pd.DataFrame(null_list_temp)
null_list_temp.reset_index(inplace=True)
null_list_temp.rename(columns={'index':'field',0:'null_entries'},inplace=True)
null_list_temp.sort_values(by=['null_entries'],inplace=True,ascending=False)

In [6]:
#add core fields that are absolutely necessary for dataset
df=df[["DevType", "LanguageHaveWorkedWith", "LanguageWantToWorkWith", "DatabaseHaveWorkedWith", "DatabaseWantToWorkWith", "YearsCode", "YearsCodePro"]]

In [7]:
#check if no of null entries are still within threshold currently 10% null rate
(df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100).round(2)

np.float64(10.89)

In [8]:
#Dropping responses by students and renaming 'others pls specify' to 'others' in DevType
df.loc[df['DevType']=='Other (please specify):',['DevType']]='Others'
df=df[(df['DevType']!='Student') & (df['DevType']!='Others')]

In [9]:
"YearsCode and YearsCodePro both have certain string values namely less than 1 year , more than 50 years we wincisor them to 0 and 51,respectively"
df.loc[df['YearsCode']=='Less than 1 year',['YearsCode']]='0'
df.loc[df['YearsCode']=='More than 50 years',['YearsCode']]='51'
df.loc[df['YearsCodePro']=='Less than 1 year',['YearsCodePro']]='0'
df.loc[df['YearsCodePro']=='More than 50 years',['YearsCodePro']]='51'

In [10]:
#handling null values in the dataset
coltran1=ColumnTransformer(transformers=[('tr1',SimpleImputer(strategy='median'),['YearsCode','YearsCodePro']),
                                        ]
                          ,remainder='passthrough')

df=coltran1.fit_transform(df)
df=pd.DataFrame(df)
df.rename(columns={0:'YearsCode',1:'YearsCodePro',2:'DevType',3:'LanguageHaveWorkedWith',4:'LanguageWantToWorkWith',5:'DatabaseHaveWorkedWith',6:'DatabaseWantToWorkWith'},inplace=True)

In [11]:
#dropping values where all four transaction columns were null as they are the main feature of this project.
df.dropna(axis=0,how='all',subset=['LanguageHaveWorkedWith','DatabaseHaveWorkedWith'],inplace=True)

In [12]:
#changing dataypes from object to int8 for years of exp and devtype as categories
coltran2=ColumnTransformer([('tr1',SimpleImputer(strategy='constant',fill_value=''),['LanguageHaveWorkedWith','LanguageWantToWorkWith','DatabaseHaveWorkedWith','DatabaseWantToWorkWith'])],remainder='passthrough')
df=coltran2.fit_transform(df)
df=pd.DataFrame(df)
df.rename(columns={0:'LanguageHaveWorkedWith',1:'LanguageWantToWorkWith',2:'DatabaseHaveWorkedWith',3:'DatabaseWantToWorkWith',4:'YearsCode',5:'YearsCodePro',6:'DevType'},inplace=True)
df=df[['YearsCode','YearsCodePro','DevType','LanguageHaveWorkedWith','LanguageWantToWorkWith','DatabaseHaveWorkedWith','DatabaseWantToWorkWith']]

In [24]:
df['YearsCode']=df['YearsCode'].astype('int8')
df['YearsCodePro']=df['YearsCodePro'].astype('int8')
df['DevType']=df['DevType'].astype('category')
df['WorkWithAgg']=df[['LanguageHaveWorkedWith','DatabaseHaveWorkedWith']].apply(lambda row: ';'.join(list(set((';'.join(x for x in row if x!='')).split(';')))),axis=1)
df['TotalAgg']=df[['LanguageHaveWorkedWith','LanguageWantToWorkWith', 'DatabaseHaveWorkedWith','DatabaseWantToWorkWith']].apply(lambda row: ';'.join(list(set((';'.join(x for x in row if x!='')).split(';')))),axis=1)

In [35]:
df.drop(columns=['LanguageHaveWorkedWith','LanguageWantToWorkWith','DatabaseHaveWorkedWith','DatabaseWantToWorkWith'],inplace=True)

In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50262 entries, 0 to 50261
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   YearsCode     50262 non-null  int8    
 1   YearsCodePro  50262 non-null  int8    
 2   DevType       50262 non-null  category
 3   WorkWithAgg   50262 non-null  str     
 4   TotalAgg      50262 non-null  str     
dtypes: category(1), int8(2), str(2)
memory usage: 933.0 KB


In [16]:
df.to_csv('current dataset.csv')